<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# EarthDaily Agriculture - Location Based Field Border

Demonstrates `LocationBasedBorderExtractor`: takes a DataFrame whose `geometry` column carries a **Point WKT** (lon,lat)
and queries the Geosys `/field-borders/v1/AutomaticBoundary` endpoint to return the field polygon containing that point.

## Step 1 — Bootstrap & Initialise

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
from earthdaily.agriculture.extractors.location_based_border_functions import LocationBasedBorderExtractor
import pandas as pd

manager = WorkflowManager('prod', log_to_console=True, log_level='DEBUG')

## Step 2 — Build a small DataFrame of points

The extractor expects a DataFrame with `id` and `geometry` columns where `geometry` is a Point WKT.
Polygons are rejected with a clear error — pre-process via `core.geometry.get_centroid_wkt()` if needed.

In [ ]:
manager.sfd_list = pd.DataFrame([
    {'id': 'loc_001', 'name': 'Field A', 'geometry': 'POINT (-98.52565383476727 41.70048113449059)'},
    {'id': 'loc_002', 'name': 'Field B', 'geometry': 'POINT (-98.45334882268624 41.65847772831499)'},
    {'id': 'loc_003', 'name': 'Field C', 'geometry': 'POINT (-98.52370557232842 41.638932729657505)'},
])
print(manager.sfd_list)

## Step 3 — Configure the extractor

In [ ]:
extractor = LocationBasedBorderExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)
extractor.setup_location_based_border_parameters(
    simplified_geom=False,   # If True, returns a simplified field geometry (lower shape-point count)
    partial_frequency=50,
)

### Test get_location_based_border_api

In [ ]:
row = manager.sfd_list.iloc[1].to_dict()
raw = extractor.get_location_based_border_api(row)
print(raw)

### Test get_location_based_border_api_safe

In [ ]:
safe = extractor.get_location_based_border_api_safe(row)
print(safe['success'], safe.get('error'))

### Test format_location_based_border_json

In [ ]:
if safe['success']:
    df = extractor.format_location_based_border_json(safe['data'])
    print(df.columns.tolist())
    print(df)

### process_single_entity_location_based_border

In [ ]:
# entity working
result = extractor.process_single_entity_location_based_border(
    pd.Series(manager.sfd_list.iloc[1])
)
if result['error']:
    print('error:', result['error'])
else:
    print(result['data'])


#error on this entity

result = extractor.process_single_entity_location_based_border(
    pd.Series(manager.sfd_list.iloc[0])
)
if result['error']:
    print('error:', result['error'])
else:
    print(result['data'])

### Polygon-input rejection

If a row carries a polygon (or any non-Point geometry), the extractor returns a clean error rather than silently degrading to a centroid.

In [ ]:
polygon_row = pd.Series({
    'id': 'bad_001',
    'geometry': 'POLYGON ((-93.6 41.5, -93.59 41.5, -93.59 41.51, -93.6 41.51, -93.6 41.5))',
})
result = extractor.process_single_entity_location_based_border(polygon_row)
print(result['error'])

## Step 4 — Bulk extraction

In [ ]:
result = extractor.process_location_based_border_bulk_extraction_parallel(
    entity_list=manager.sfd_list,
    max_workers=5,
    output_path=manager.output_result_dir,
    prefix='location_borders',
)
print(f'Processed: {result["successful_calculations"]}/{result["total_calculations"]}')
print(result['results_df'].columns.tolist())

In [ ]:
result['results_df']

### Convert to a GeoDataFrame

Once you have polygons as WKT, convert to a GeoDataFrame to plot or write to GPKG/parquet.

In [ ]:
import geopandas as gpd
from shapely import wkt

df = result['results_df'].copy()
df['geometry'] = df['polygon_geometry'].apply(lambda w: wkt.loads(w) if isinstance(w, str) else None)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
gdf